# Spacing Statistics - Oneline

## 1. Importing / Installing Packages

In [1]:
import os # Importing os module for operating system dependent functionality

import glob # Importing glob module for file pattern matching

import pandas as pd # Importing pandas package

# Set the maximum number of columns to display to None
pd.set_option('display.max_columns', None)

import numpy as np # Importing numpy package

from typing import Dict, Tuple, List, Union, Optional, ClassVar, Any, Iterable, Literal # Importing specific types from typing module

# from src.utils import DatabricksOdbcConnector # Importing DatabricksOdbcConnector class from database_manager module

# from tqdm import tqdm # Importing tqdm for progress bar functionality

# from joblib import Parallel, delayed # Importing Parallel and delayed for parallel processing

# from matplotlib import pyplot as plt # Importing pyplot from matplotlib for plotting

# from pyproj import Geod # Importing Geod class from pyproj for geodetic calculations

# Setting matplotlib to inline mode for Jupyter notebooks
%matplotlib inline

%config InlineBackend.figure_format = 'svg' # Configuring inline backend to use SVG format for figures

# from src.well_data import WellDataLoader, GeoSurveyProcessor # Importing custom classes for well data management

# from __future__ import annotations # Enabling future annotations for type hinting

from src.utils import reorder_columns # Importing utility function to reorder DataFrame columns

from __future__ import annotations

## 2. Importing Data to Dataframes

### 2.1 Importing Spacing i-k pairs stat data to dataframe

In [2]:
# Path to your folder
folder_path = r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\notebooks\spacing_batches_MB"

# Get all parquet files in the folder
parquet_files = glob.glob(os.path.join(folder_path, "*.parquet"))

# Read and concatenate them into a single DataFrame
df_spacing_ik = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

In [3]:
df_spacing_ik_filterd = df_spacing_ik[df_spacing_ik['reject_reason']==""].reset_index(drop=True).copy()

In [4]:
df_spacing_ik_filterd = reorder_columns(df=df_spacing_ik_filterd, columns_to_move=['direction_axis', 'direction_to_k_from_i_axis',
       'direction_axis_confidence', 'direction_axis_distribution',
       'axis_forced'], reference_column='drill_direction_k')

### 2.2 Importing Header Data

In [9]:
df_header = pd.read_excel(
    r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\data\MB_Header_Bench_Edits.xlsx",
                           sheet_name='MB_Header_CC', dtype={'uwi': str, 'API10': str})

df_header.drop(columns=['bench_cc', 'bench_novi', 'surface_lat', 'surface_lon'], inplace=True)

In [10]:
df_header.head()

,API10,uwi,lease_name,well_name,well_num,operator,rsv_cat,bench,first_prod_date,comp_date,hole_direction
0,4231743164,42317431640000,HAYDEN 14-23-A,HAYDEN 14-23-A 4101H,4101H,XOM,01PDP,WCA,2021-06-01,2021-06-29,H
1,4231742907,42317429070000,JACK MOHR UNIT 2,JACK MOHR UNIT 2 0266DH,0266DH,XOM,01PDP,Dean,2022-02-01,2022-02-20,H
2,4231743195,42317431950000,YOU JANE 20-25 UNIT 1,YOU JANE 20-25 UNIT 1 112,112,FANG,01PDP,LSS,2021-04-01,2021-04-28,H
3,4231743422,42317434220000,MINOTAUR 38-66 UNIT 1,MINOTAUR 38-66 UNIT 1 142,142,FANG,01PDP,LSS,2022-01-01,2022-01-07,H
4,4231739955,42317399550100,MABEE J E `B` NCT-2,MABEE J E `B` NCT-2 1508H,1508H,COP,01PDP,WCB,2015-07-01,2015-07-15,H


## 3. Define the Class

In [11]:
Direction = Literal["E", "W", "N", "S"]
AxisMode = Literal["any", "EW", "NS"]
AxisPref = Literal["EW", "NS"]


class DirectionalBenchNeighbors:
    """
    Summarize nearest neighbors per well_i by bench and opposite direction.

    For each well_i, produce:
      - SAME-bench closest (any direction, or axis-limited) -> *_same_1
      - SAME-bench closest in the OPPOSITE direction of same_1 -> *_same_2
      - DIFFERENT-bench closest (any direction, or axis-limited) -> *_near_1
      - DIFFERENT-bench closest in the OPPOSITE direction of near_1 -> *_near_2

    Required columns
    ----------------
    spacing_df:
        'well_i', 'well_k', 'horizontal_dist', 'vertical_dist', '3D_dist',
        'direction_to_k_from_i_axis'  # exactly one of {'E','W','N','S'}
    header_df:
        'uwi', 'bench'

    Parameters (at call-time)
    -------------------------
    cutoff_ft : float
        Maximum allowed horizontal distance (feet).
    axis_mode : {'any', 'EW', 'NS'}, default 'any'
        If 'EW', only E/W directions are eligible for the *_1 pick.
        If 'NS', only N/S directions are eligible for the *_1 pick.
        If 'any', all four directions can compete for *_1.
        (Note: *_2 is always the opposite of *_1’s direction.)
    prefer_axis : {'EW', 'NS'} | None, default None
        If set and there is a tie in `horizontal_dist` for *_1 across axes,
        prefer the candidate whose direction is in the given axis.
        (Only affects *_1; *_2 is determined solely by the opposite direction.)

    Output columns
    --------------
      uwi_same_1,  hz_ft_to_same_1,  vt_ft_to_same_1,  3d_ft_to_same_1,
      uwi_same_2,  hz_ft_to_same_2,  vt_ft_to_same_2,  3d_ft_to_same_2,
      uwi_near_1,  hz_ft_to_near_1,  vt_ft_to_near_1,  3d_ft_to_near_1,
      uwi_near_2,  hz_ft_to_near_2,  vt_ft_to_near_2,  3d_ft_to_near_2

    Examples
    --------
    >>> nb = DirectionalBenchNeighbors()
    >>> # Example 1: cutoff=1320 ft, no axis restriction, no preference
    >>> out1 = nb.summarize(spacing_df, header_df, cutoff_ft=1320.0)
    >>> out1.filter(regex="^uwi_|^hz_ft_").head()

    >>> # Example 2: restrict initial pick to EW and prefer EW over NS on ties
    >>> out2 = nb.summarize(spacing_df, header_df, cutoff_ft=1320.0,
    ...                     axis_mode="EW", prefer_axis="EW")

    >>> # Example 3: allow any axis but prefer NS on ties
    >>> out3 = nb.summarize(spacing_df, header_df, cutoff_ft=1320.0,
    ...                     axis_mode="any", prefer_axis="NS")
    """

    _OPPOSITE: Dict[Direction, Direction] = {"E": "W", "W": "E", "N": "S", "S": "N"}
    _ALL_DIRS: Tuple[Direction, ...] = ("E", "W", "N", "S")

    def __init__(self, *, tie_break_on: str = "well_k") -> None:
        """
        Parameters
        ----------
        tie_break_on : str, default 'well_k'
            Secondary stable key when distances tie.
        """
        self.tie_break_on = tie_break_on

    # ---------- Public API ----------

    def summarize(
        self,
        spacing_df: pd.DataFrame,
        header_df: pd.DataFrame,
        *,
        cutoff_ft: float,
        axis_mode: AxisMode = "any",
        prefer_axis: Optional[AxisPref] = None,
    ) -> pd.DataFrame:
        """Return one summary row per well_i with *_same_{1,2} and *_near_{1,2}.

        See class docstring for details on behavior and examples.

        Raises
        ------
        ValueError
            If required columns are missing or inputs are inconsistent.
        """
        self._validate_inputs(spacing_df, header_df)

        # Normalize IDs for deterministic merges/sorts
        spacing = spacing_df.copy()
        spacing["well_i"] = spacing["well_i"].astype(str)
        spacing["well_k"] = spacing["well_k"].astype(str)
        header = header_df.copy()
        header["uwi"] = header["uwi"].astype(str)

        # Map benches
        bench_map: Dict[str, str] = header.set_index("uwi")["bench"].to_dict()
        spacing["bench_i"] = spacing["well_i"].map(bench_map)
        spacing["bench_k"] = spacing["well_k"].map(bench_map)

        # Filter by cutoff and valid directions (W/E/N/S guaranteed by your data)
        spacing = spacing.loc[spacing["horizontal_dist"] <= cutoff_ft].copy()

        if spacing.empty:
            wells = spacing_df["well_i"].astype(str).unique()
            return self._empty_summary(wells)

        # SAME vs NEAR splits
        spacing["is_same"] = spacing["bench_i"] == spacing["bench_k"]
        same = spacing.loc[spacing["is_same"]].copy()
        near = spacing.loc[~spacing["is_same"]].copy()

        # Compute summaries with axis-mode & preference logic
        same_summary = self._compute_category_summary(
            same, category="same", axis_mode=axis_mode, prefer_axis=prefer_axis
        )
        near_summary = self._compute_category_summary(
            near, category="near", axis_mode=axis_mode, prefer_axis=prefer_axis
        )

        # Merge to one row per well_i (include wells with no candidates)
        all_wells = (
            spacing_df["well_i"].astype(str).drop_duplicates().to_frame(name="well_i")
        )
        out = (
            all_wells.merge(same_summary, on="well_i", how="left")
                     .merge(near_summary, on="well_i", how="left")
        )

        # Column order
        ordered_cols = (
            ["well_i"]
            + self._category_cols("same", 1)
            + self._category_cols("same", 2)
            + self._category_cols("near", 1)
            + self._category_cols("near", 2)
        )
        existing_cols = [c for c in ordered_cols if c in out.columns]
        remaining_cols = [c for c in out.columns if c not in existing_cols]
        return out[existing_cols + remaining_cols]

    # ---------- Internals: category computation ----------

    def _compute_category_summary(
        self,
        df: pd.DataFrame,
        *,
        category: Literal["same", "near"],
        axis_mode: AxisMode,
        prefer_axis: Optional[AxisPref],
    ) -> pd.DataFrame:
        """
        Build a summary for one category (same/near) using vectorized reductions.

          1) Best-per-direction (well_i, direction) by horizontal_dist
             (tie-break: well_k for determinism).
          2) *_1: among eligible directions (based on axis_mode), pick best overall.
             If `prefer_axis` is set, bias ties toward that axis family.
          3) *_2: pick best entry in the opposite direction of *_1 (if exists).

        Returns
        -------
        pd.DataFrame with:
          ['well_i'] + uwi_{cat}_1, hz_ft_to_{cat}_1, vt_ft_to_{cat}_1, 3d_ft_to_{cat}_1,
                       uwi_{cat}_2, hz_ft_to_{cat}_2, vt_ft_to_{cat}_2, 3d_ft_to_{cat}_2
        """
        if df.empty:
            return pd.DataFrame(columns=["well_i"] + self._category_cols(category, 1) + self._category_cols(category, 2))

        # Step 1: best per-direction (argmin via sort + drop_duplicates)
        per_dir = (
            df.assign(direction=df["direction_to_k_from_i_axis"])
              .sort_values(["well_i", "direction", "horizontal_dist", self.tie_break_on])
              .drop_duplicates(subset=["well_i", "direction"], keep="first")
              .loc[:, ["well_i", "direction", "well_k", "horizontal_dist", "vertical_dist", "3D_dist"]]
        )

        if per_dir.empty:
            return pd.DataFrame(columns=["well_i"] + self._category_cols(category, 1) + self._category_cols(category, 2))

        # Eligible directions for *_1 based on axis_mode
        if axis_mode == "EW":
            eligible = {"E", "W"}
        elif axis_mode == "NS":
            eligible = {"N", "S"}
        else:
            eligible = {"E", "W", "N", "S"}

        per_dir_eligible = per_dir.loc[per_dir["direction"].isin(eligible)].copy()
        if per_dir_eligible.empty:
            # No eligible directions for *_1 => no *_1 and consequently no *_2
            return pd.DataFrame({"well_i": per_dir["well_i"].unique()}).astype({"well_i": str})

        # Step 2: choose *_1 across eligible directions with optional axis preference
        if prefer_axis is None:
            # Pure distance, then well_k
            best_overall = (
                per_dir_eligible.sort_values(["well_i", "horizontal_dist", "well_k"])
                                .drop_duplicates(subset=["well_i"], keep="first")
            )
        else:
            prefer_set = {"E", "W"} if prefer_axis == "EW" else {"N", "S"}
            # Priority 0 if in preferred axis family; 1 otherwise
            per_dir_eligible = per_dir_eligible.assign(
                _axis_priority=np.where(per_dir_eligible["direction"].isin(prefer_set), 0, 1)
            )
            best_overall = (
                per_dir_eligible.sort_values(["well_i", "horizontal_dist", "_axis_priority", "well_k"])
                                .drop_duplicates(subset=["well_i"], keep="first")
            )

        best_overall = best_overall.rename(columns={
            "well_k": f"uwi_{category}_1",
            "horizontal_dist": f"hz_ft_to_{category}_1",
            "vertical_dist": f"vt_ft_to_{category}_1",
            "3D_dist": f"3d_ft_to_{category}_1",
            "direction": f"direction_{category}_1",
        })

        keep_cols_1 = ["well_i",
                       f"uwi_{category}_1",
                       f"hz_ft_to_{category}_1",
                       f"vt_ft_to_{category}_1",
                       f"3d_ft_to_{category}_1",
                       f"direction_{category}_1"]
        best_overall = best_overall.loc[:, keep_cols_1]

        if best_overall.empty:
            return best_overall.drop(columns=[f"direction_{category}_1"], errors="ignore")

        # Step 3: *_2 is best in the opposite direction of *_1 (if present)
        best_overall = best_overall.copy()
        best_overall[f"opp_dir_{category}"] = best_overall[f"direction_{category}_1"].map(self._OPPOSITE)

        # Build lookup for opposite-direction minima (from all per_dir, not just eligible)
        per_dir_for_merge = per_dir.rename(columns={
            "direction": f"opp_dir_{category}",
            "well_k": f"uwi_{category}_2",
            "horizontal_dist": f"hz_ft_to_{category}_2",
            "vertical_dist": f"vt_ft_to_{category}_2",
            "3D_dist": f"3d_ft_to_{category}_2",
        })

        merged = best_overall.merge(
            per_dir_for_merge[["well_i",
                               f"opp_dir_{category}",
                               f"uwi_{category}_2",
                               f"hz_ft_to_{category}_2",
                               f"vt_ft_to_{category}_2",
                               f"3d_ft_to_{category}_2"]],
            on=["well_i", f"opp_dir_{category}"],
            how="left",
        )

        return merged.drop(columns=[f"direction_{category}_1", f"opp_dir_{category}"])

    # ---------- Internals: utilities ----------

    @staticmethod
    def _category_cols(cat: Literal["same", "near"], idx: Literal[1, 2]) -> List[str]:
        return [
            f"uwi_{cat}_{idx}",
            f"hz_ft_to_{cat}_{idx}",
            f"vt_ft_to_{cat}_{idx}",
            f"3d_ft_to_{cat}_{idx}",
        ]

    @staticmethod
    def _empty_summary(wells: Iterable[str]) -> pd.DataFrame:
        cols = ["well_i"]
        for cat in ("same", "near"):
            for idx in (1, 2):
                cols += [
                    f"uwi_{cat}_{idx}",
                    f"hz_ft_to_{cat}_{idx}",
                    f"vt_ft_to_{cat}_{idx}",
                    f"3d_ft_to_{cat}_{idx}",
                ]
        out = pd.DataFrame({"well_i": list(map(str, wells))})
        for c in cols:
            if c != "well_i":
                out[c] = np.nan
        return out

    @staticmethod
    def _validate_inputs(spacing_df: pd.DataFrame, header_df: pd.DataFrame) -> None:
        spacing_required = {
            "well_i",
            "well_k",
            "horizontal_dist",
            "vertical_dist",
            "3D_dist",
            "direction_to_k_from_i_axis",
        }
        header_required = {"uwi", "bench"}

        missing_s = spacing_required - set(spacing_df.columns)
        missing_h = header_required - set(header_df.columns)
        if missing_s:
            raise ValueError(f"spacing_df missing required columns: {sorted(missing_s)}")
        if missing_h:
            raise ValueError(f"header_df missing required columns: {sorted(missing_h)}")

## 3. Testing

In [17]:
# Joining benches with spacing data

df_spacing_filt_bench = df_spacing_ik_filterd.copy()

bench_map = df_header.set_index("uwi")["bench"]

df_spacing_filt_bench["bench_i"] = df_spacing_filt_bench["well_i"].map(bench_map)
df_spacing_filt_bench["bench_k"] = df_spacing_filt_bench["well_k"].map(bench_map)

df_spacing_filt_bench = reorder_columns(df=df_spacing_filt_bench, columns_to_move=['bench_i', 'bench_k'], reference_column='well_k')

In [15]:
nb = DirectionalBenchNeighbors()

# A) Classic: 1320′ cutoff, any axis, no preference
res_any = nb.summarize(spacing_df=df_spacing_ik_filterd, header_df=df_header, cutoff_ft=1320.0)

# B) Only EW is eligible for *_1, and prefer EW if ties occur
res_ew = nb.summarize(spacing_df=df_spacing_ik_filterd, header_df=df_header, cutoff_ft=1320.0, axis_mode="EW", prefer_axis="EW")

# C) Any axis is eligible for *_1, but prefer NS on distance ties
res_any_pref_ns = nb.summarize(spacing_df=df_spacing_ik_filterd, header_df=df_header, cutoff_ft=1320.0, axis_mode="any", prefer_axis="NS")


In [16]:
res_any

,well_i,uwi_same_1,hz_ft_to_same_1,vt_ft_to_same_1,3d_ft_to_same_1,uwi_same_2,hz_ft_to_same_2,vt_ft_to_same_2,3d_ft_to_same_2,uwi_near_1,hz_ft_to_near_1,vt_ft_to_near_1,3d_ft_to_near_1,uwi_near_2,hz_ft_to_near_2,vt_ft_to_near_2,3d_ft_to_near_2
0,42003452020000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42003473970000,681.547994,220.1850,716.232576,NaN,NaN,NaN,NaN
1,42003455530000,42003480110000,1054.700601,33.0405,1055.218003,42003488960000,1097.633089,24.27,1097.901376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,42003455740000,42003489240000,1130.518783,9.7600,1130.560912,42003489230000,1152.432141,10.26,1152.477812,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,42003455960000,42003457220000,1209.862543,23.4050,1210.088909,42003456210000,1236.511254,5.68,1236.524300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,42003456210000,42003470810000,190.757107,41.4000,195.197935,42003455960000,1236.533185,5.68,1236.546231,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24203,42501375570000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501375580000,653.745825,6.9750,653.783033,NaN,NaN,NaN,NaN
24204,42501375580000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501375570000,653.751267,6.9750,653.788475,NaN,NaN,NaN,NaN
24205,42501375980000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501373630000,522.599090,0.8330,522.599754,42501376000000,1314.917635,2.495,1314.920002
24206,42501376000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501376030000,661.474624,4.3550,661.488960,42501375980000,1314.909453,2.495,1314.911821


In [ ]:
df_spacing_filt_bench

,well_i,well_k,bench_i,bench_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,drill_direction_i,drill_direction_k,direction_axis,direction_to_k_from_i_axis,direction_axis_confidence,direction_axis_distribution,axis_forced,overlap_len_ft,n_samples,dy_p5,angle_deg,pair_alignment,min_distance_ft,mean_windowed_ft,reject_reason
0,42003452020000,42003461880000,CLFK,CLFK,2668.066503,2674.946793,35.1850,2668.298493,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,4254.688290,43.0,2589.688596,1.705439,parallel_like,NaN,NaN,
1,42003452020000,42003472060000,CLFK,WICHITA ALBANY,1390.051959,1387.517452,366.6700,1437.599157,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,4308.840578,44.0,1348.740326,0.621586,parallel_like,NaN,NaN,
2,42003452020000,42003472070000,CLFK,WICHITA ALBANY,2661.911494,2662.095167,418.2400,2694.568147,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,4302.962458,44.0,2592.452940,1.680333,parallel_like,NaN,NaN,
3,42003452020000,42003473240000,CLFK,WICHITA ALBANY,1853.216683,1857.706734,329.0150,1882.196309,NS,NS,EW,E,1.0,"E:1.00,W:0.00",True,4451.992349,45.0,1765.128339,1.899251,parallel_like,NaN,NaN,
4,42003452020000,42003473970000,CLFK,TUBB,681.547994,679.350945,220.1850,716.232576,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,4185.335375,42.0,609.608285,1.562277,parallel_like,NaN,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660813,42501376030000,42501373660000,NaN,SAN ANDRES,433.048135,435.712969,20.7925,433.547017,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,1231.775724,13.0,389.713407,2.983424,parallel_like,NaN,NaN,
660814,42501376030000,42501374060000,NaN,SAN ANDRES,3657.013955,3658.190471,14.6150,3657.043159,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,1199.165510,12.0,3616.277759,2.581051,parallel_like,NaN,NaN,
660815,42501376030000,42501374070000,NaN,SAN ANDRES,3056.343786,3055.010513,7.2705,3056.352433,NS,NS,EW,W,1.0,"E:0.00,W:1.00",True,1213.071038,13.0,3026.641634,3.042419,parallel_like,NaN,NaN,
660816,42501376030000,42501375980000,NaN,NaN,1976.414583,1976.376924,1.8600,1976.415458,NS,NS,EW,E,1.0,"E:1.00,W:0.00",True,5032.011912,51.0,1957.358225,0.071674,parallel_like,NaN,NaN,
